# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata: .name and .description properties
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields
print("Available record sets and fields:")
record_sets = [rs for rs in dataset.metadata.record_sets]
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']} | Name: {rs.get('name', '[no name]')}")
    print("  Fields:")
    for field in rs.get('fields', []):
        field_id = field.get('@id','')
        field_name = field.get('name', '[no name]')
        field_dt = field.get('dataType', '[no type]')
        print(f"    - Field @id: {field_id}	Name: {field_name}	Type: {field_dt}")
    print("")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. 
Use the record set and field `@id`s from the overview above.

In [ ]:
# Choose record sets for extraction (using @id)
# For this dataset, the main record set is: 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034.records'
main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034.records'
record_sets_ids = [main_record_set_id]
dataframes = {}

for rs_id in record_sets_ids:
    print(f"Loading records for record set: {rs_id}")
    # records() yields dicts for each row in record set
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

print("Fields in main record set DataFrame:")
print(dataframes[main_record_set_id].columns.tolist())

# Show the first 5 records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Identify a suitable numeric field from the DataFrame columns
numeric_field_id = None
for col in dataframes[main_record_set_id].columns:
    if ('coverage' in col.lower()) or ('percent' in col.lower()) or ('abundance' in col.lower()) or (col.lower() == 'mw'):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # fallback to first float/integer-like column
    for col in dataframes[main_record_set_id].columns:
        if pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][col]):
            numeric_field_id = col
            break
if numeric_field_id is None:
    raise ValueError("No numeric field found in main record set for EDA.")
print(f"Selected numeric field for EDA: {numeric_field_id}")

# Example threshold, change as appropriate
threshold = dataframes[main_record_set_id][numeric_field_id].mean()

filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field if available
group_field = None
# Find a field for grouping (e.g., sample/condition/type/description)
for col in dataframes[main_record_set_id].columns:
    if (col.lower() != numeric_field_id.lower()) and (
        any(k in col.lower() for k in ['sample', 'condition', 'type', 'description'])
    ):
        group_field = col
        break
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} grouped by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field found. Skipping group-by step.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), bins=30, kde=True, color='skyblue')
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If group_field is found, boxplot grouped by that field (top 10 groups only)
if group_field is not None:
    plt.figure(figsize=(10, 5))
    order = filtered_df[group_field].value_counts().index[:10]
    sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df, order=order)
    plt.title(f'{numeric_field_id} by {group_field} (top 10 groups)')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

This notebook detailed how to load, inspect, and process the FAIR<sup>2</sup> dataset:
- Loaded dataset metadata and reviewed the available record sets and field `@id`s
- Extracted the main record set (using its `@id`) as a DataFrame
- Applied basic EDA, filtering, normalization, and optionally grouped by sample/condition fields where present
- Visualized numeric distributions and relationships

This workflow provides a foundation for more advanced analyses, modeling, or integration with domain-specific workflows using the `mlcroissant` library and referencing all data entities by their Croissant `@id`.